# Experiment 4: Bayesian Network
**Name:** Arshan Attar  
**Roll Number:** 231408  


In [ ]:
pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 10.7 MB/s eta 0:00:00


In [ ]:
# Install the required library (Run this in your terminal or uncomment if running in a notebook)
# !pip install pgmpy

import pgmpy.models
import pgmpy.inference
import pgmpy.factors.discrete
import networkx as nx
import pylab as plt

# 1. Create a Bayesian Network structure
# Define the nodes and directed edges (Parent -> Child)
model = pgmpy.models.DiscreteBayesianNetwork([
    ('Burglary', 'Alarm'),
    ('Earthquake', 'Alarm'),
    ('Alarm', 'JohnCalls'),
    ('Alarm', 'MaryCalls')
])

# 2. Define Conditional Probability Distributions (CPDs)
# Note: In pgmpy, state 0 is typically True and state 1 is False for this specific setup based on your arrays.
# It's highly recommended to map states explicitly if needed, but we will follow your array structure.

# Probability of burglary: P(Burglary)
# [[P(B=True)], [P(B=False)]]
cpd_burglary = pgmpy.factors.discrete.TabularCPD('Burglary', 2, [[0.001], [0.999]])

# Probability of earthquake: P(Earthquake)
# [[P(E=True)], [P(E=False)]]
cpd_earthquake = pgmpy.factors.discrete.TabularCPD('Earthquake', 2, [[0.002], [0.998]])

# Probability of alarm going off given a burglary and/or earthquake: P(Alarm | Burglary, Earthquake)
# Columns represent combinations of evidence (B=T/E=T, B=T/E=F, B=F/E=T, B=F/E=F)
cpd_alarm = pgmpy.factors.discrete.TabularCPD(
    'Alarm', 2,
    [[0.95, 0.94, 0.29, 0.001],  # P(Alarm=True)
     [0.05, 0.06, 0.71, 0.999]], # P(Alarm=False)
    evidence=['Burglary', 'Earthquake'],
    evidence_card=[2, 2]
)

# Probability that John calls given the alarm has sounded: P(JohnCalls | Alarm)
cpd_john = pgmpy.factors.discrete.TabularCPD(
    'JohnCalls', 2,
    [[0.90, 0.05],  # P(John=True)
     [0.10, 0.95]], # P(John=False)
    evidence=['Alarm'],
    evidence_card=[2]
)

# Probability that Mary calls given the alarm has sounded: P(MaryCalls | Alarm)
cpd_mary = pgmpy.factors.discrete.TabularCPD(
    'MaryCalls', 2,
    [[0.70, 0.01],  # P(Mary=True)
     [0.30, 0.99]], # P(Mary=False)
    evidence=['Alarm'],
    evidence_card=[2]
)

# 3. Add CPDs to the network structure
model.add_cpds(cpd_burglary, cpd_earthquake, cpd_alarm, cpd_john, cpd_mary)

# 4. Check if the model is valid (Verifies if probabilities sum to 1)
model.check_model()

# 5. Print probability distributions to verify
print('Probability distribution, P(Burglary)')
print(cpd_burglary, '\n')

print('Probability distribution, P(Earthquake)')
print(cpd_earthquake, '\n')

print('Joint probability distribution, P(Alarm | Burglary, Earthquake)')
print(cpd_alarm, '\n')

print('Joint probability distribution, P(JohnCalls | Alarm)')
print(cpd_john, '\n')

print('Joint probability distribution, P(MaryCalls | Alarm)')
print(cpd_mary, '\n')

# 6. Perform variable elimination for inference
# Variable elimination (VE) is an exact inference algorithm in bayesian networks
infer = pgmpy.inference.VariableElimination(model)

# Query 1: Calculate the posterior probability of a burglary if John and Mary call (0: True, 1: False)
posterior_burglary = infer.query(['Burglary'], evidence={'JohnCalls': 0, 'MaryCalls': 0})
print('Posterior probability of Burglary given JohnCalls=True and MaryCalls=True')
print(posterior_burglary, '\n')

# Query 2: Calculate the posterior probability of alarm sounding if there is a burglary and an earthquake
posterior_alarm = infer.query(['Alarm'], evidence={'Burglary': 0, 'Earthquake': 0})
print('Posterior probability of Alarm given Burglary=True and Earthquake=True')
print(posterior_alarm, '\n')

Probability distribution, P(Burglary)
+-------------+-------+
| Burglary(0) | 0.001 |
+-------------+-------+
| Burglary(1) | 0.999 |
+-------------+-------+ 

Probability distribution, P(Earthquake)
+---------------+-------+
| Earthquake(0) | 0.002 |
+---------------+-------+
| Earthquake(1) | 0.998 |
+---------------+-------+ 

Joint probability distribution, P(Alarm | Burglary, Earthquake)
+------------+---------------+---------------+---------------+---------------+
| Burglary   | Burglary(0)   | Burglary(0)   | Burglary(1)   | Burglary(1)   |
+------------+---------------+---------------+---------------+---------------+
| Earthquake | Earthquake(0) | Earthquake(1) | Earthquake(0) | Earthquake(1) |
+------------+---------------+---------------+---------------+---------------+
| Alarm(0)   | 0.95          | 0.94          | 0.29          | 0.001         |
+------------+---------------+---------------+---------------+---------------+
| Alarm(1)   | 0.05          | 0.06          | 0.71

### **Solution to the Stated Problem**

**Problem:** Calculate the probability that an alarm has sounded, but there is neither a burglary, nor an earthquake occurred, and John and Mary both called Harry.

Mathematically, we need to find the joint probability: $P(John, Mary, A, \bar{B}, \bar{E})$ which translates to $P(J=T, M=T, A=T, B=F, E=F)$.

Based on the Bayesian Network's conditional independence, we can calculate this joint probability by multiplying the individual probabilities from the tables:

$$P(J=T, M=T, A=T, B=F, E=F) = P(B=F) \times P(E=F) \times P(A=T | B=F, E=F) \times P(J=T | A=T) \times P(M=T | A=T)$$

Substituting the values from the provided CPD tables:

*   $P(B=F) = 0.999$
*   $P(E=F) = 0.998$
*   $P(A=T | B=F, E=F) = 0.001$
*   $P(J=T | A=T) = 0.90$
*   $P(M=T | A=T) = 0.70$


**Calculation:**

$$0.999 \times 0.998 \times 0.001 \times 0.90 \times 0.70 = 0.00062811114$$

**Final Answer:**
The probability that the alarm sounds without a burglary or earthquake, and both John and Mary call, is approximately **0.000628** (or about **0.063%**).